# voice-rs — huấn luyện adapter trên Colab (CPU)

Luồng: cài Rust → clone repo → **tải bundle model từ GitHub Release** (`scripts/fetch_assets.sh`)
→ chạy `scripts/run_training.sh` (build → validation → training) → tải kết quả về.

Bundle chỉ chứa MODEL: trọng số `dec` đã export + ContentVec ONNX, nên
**Colab không cần PyTorch/export** — chỉ Rust.

Chạy trong **Runtime → Change runtime type → CPU** (chưa hỗ trợ GPU).

⚠️ chi phí: Colab free ~2 core. `SEG=86` (1 s) ≈ 1.5–2 h cho 250 bước.
Muốn cả câu (589 frame) thì đặt `SEG=589` — chậm hơn ~7×.


## 1. Cài Rust


In [ ]:
%%bash
set -e
if ! command -v cargo >/dev/null 2>&1; then
  curl --proto "=https" --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal
fi
source $HOME/.cargo/env
cargo --version && rustc --version


## 2. Clone repo


In [ ]:
%%bash
set -e
cd /content
[ -d voice-rs ] || git clone https://github.com/hungpham10/voice-rs.git
cd voice-rs && git log --oneline -1


## 3. Tải bundle model từ Release `assets-v1`
`voice-assets.tar.gz` = ContentVec ONNX + `vits_ljs_dec` (VITS generator đã export).


In [ ]:
%%bash
set -e
cd /content/voice-rs
./scripts/fetch_assets.sh assets-v1


## 4. Training + validation
`SEG` = frame latent; `STEPS`, `LR` tuỳ chỉnh qua biến môi trường.


In [ ]:
%%bash
set -e
cd /content/voice-rs
source $HOME/.cargo/env
SEG=86 STEPS=250 LR=1e-3 ./scripts/run_training.sh 2>&1 | tee /content/train.log


## 5. Tải kết quả về máy


In [ ]:
from google.colab import files
files.download("/content/voice-rs/test_output.wav")
files.download("/content/voice-rs/sources/features/adapter.bpk")
